# Qwen3.5-0.8B: nonlinear recurrent readout

Test the proposed **PDelta3-GDN2-CLVR + bounded local attention + shared nonlinear readout**.
Compare the original model, an adapted full-attention control, and **0 / 1 / 2 refinement steps**.

Select **Runtime → Change runtime type → GPU**, then **Run all**. The default balanced run
uses three replacement layers and jointly trains them. Choose `smoke` for a pipeline check,
or `extended` for a larger screening run. Google Drive stores checkpoints, resumable optimizer
state, exact token blocks, charts and reports. No automatic Hugging Face publication.

This is a new experiment, not a validated model. The recurrent update uses a PyTorch reference
implementation; fused CUDA/Triton kernels are future work. **2× speed is measured, never assumed.**
Qwen3.5 is already a hybrid architecture: its native linear-attention layers stay intact.

In [ ]:
import os, sys, subprocess, json, shutil
from pathlib import Path
from datetime import datetime, timezone
REPO = Path('/content/TinyCeNN-LM-nonlinear')
REPO_REF = 'main'  # A commit SHA can be used to reproduce a specific implementation.
if not REPO.exists():
    subprocess.run(['git','clone','https://github.com/vtavakkoli/TinyCeNN-LM.git',str(REPO)],check=True)
subprocess.run(['git','fetch','origin',REPO_REF],cwd=REPO,check=True)
subprocess.run(['git','checkout','--detach','FETCH_HEAD'],cwd=REPO,check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','transformers==5.17.0',
                'datasets==4.8.5','sentencepiece','pandas','matplotlib','nbformat'],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','-e',str(REPO),'--no-deps'],check=True)
print('Source commit:', subprocess.check_output(['git','rev-parse','HEAD'],cwd=REPO,text=True).strip())
# Training runs in a fresh subprocess, so imported notebook packages cannot keep an older Transformers API.

The default kernel setup is portable and can be slow. On a supported CUDA GPU, advanced users
may install compatible `flash-linear-attention` and `causal-conv1d` versions before launching
the Qwen run. The report records fallback warnings and withholds the requested-target flag
if the Qwen baseline used slow reference kernels. Do not compare this run against published
serving throughput on different hardware.

In [ ]:
FAMILY = 'qwen35'
PROFILE = 'balanced'  # 'smoke', 'balanced', or 'extended'
LAYERS = '3,7,11'
FEATURES = 64         # Try 96 in a separate run, using fresh final evaluation documents.
LOCAL_WINDOW = 32
REFINEMENT_RANK = 16
STATE_DTYPE = 'fp32'  # FP16 state is experimental; the cache-equivalence gate still applies.
RESUME = False
RUN_NAME = ''         # To resume: paste the printed run name here and set RESUME=True.
EXCLUDE_MANIFESTS = [] # Prior experiment manifest.json paths; excludes every listed document hash.

profiles = {
 'smoke': dict(train_contexts='64',test_contexts='64,128',train_documents=8,validation_documents=4,
               test_documents=4,warm_steps=1,joint_steps=2,eval_every=1,decode_tokens=8,
               timing_documents=1,timing_repeats=1),
 'balanced': dict(train_contexts='256,512',test_contexts='256,512,1024',train_documents=96,
               validation_documents=16,test_documents=32,warm_steps=20,joint_steps=150,
               eval_every=25,decode_tokens=32,timing_documents=3,timing_repeats=3),
 'extended': dict(train_contexts='256,512,1024',test_contexts='512,1024,2048',train_documents=512,
               validation_documents=32,test_documents=64,warm_steps=50,joint_steps=800,
               eval_every=50,decode_tokens=64,timing_documents=5,timing_repeats=5),
}
if PROFILE not in profiles:
    raise ValueError('Choose smoke, balanced, or extended')
if RESUME and not RUN_NAME:
    raise ValueError('Paste the previous RUN_NAME to resume exactly that experiment')
if not RUN_NAME:
    RUN_NAME = PROFILE + '-' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
from google.colab import drive
drive.mount('/content/drive')
OUT = Path('/content/drive/MyDrive/TinyCeNN-LM') / (FAMILY+'-nonlinear-readout') / RUN_NAME
OUT.parent.mkdir(parents=True,exist_ok=True)
print('RUN_NAME =', RUN_NAME)
print('Results and checkpoints:', OUT)

In [ ]:
# GPU preflight in a fresh process; never silently train on CPU in Colab.
subprocess.run([sys.executable,'-c',
    "import torch; assert torch.cuda.is_available(), 'Select a GPU runtime'; "
    "print('GPU:', torch.cuda.get_device_name()); print('PyTorch:',torch.__version__)"],check=True)
cmd = [sys.executable,'-u',str(REPO/'scripts/benchmark_nonlinear_readout.py'),
       '--family',FAMILY,'--layers',LAYERS,'--features',str(FEATURES),'--window',str(LOCAL_WINDOW),
       '--rank',str(REFINEMENT_RANK),'--refinement-steps','0,1,2','--state-dtype',STATE_DTYPE,
       '--output-dir',str(OUT)]
for key,value in profiles[PROFILE].items():
    cmd.extend(['--'+key.replace('_','-'),str(value)])
for path in EXCLUDE_MANIFESTS:
    cmd.extend(['--exclude-manifest',str(path)])
if RESUME:
    cmd.append('--resume')
# Keep stdout live and preserve stderr (including kernel fallbacks) in the run log.
log_path = OUT.parent / (RUN_NAME+'.log')
process = subprocess.Popen(cmd,cwd=REPO,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
try:
    with log_path.open('a') as log:
        for line in process.stdout:
            print(line,end='')
            log.write(line)
            log.flush()
    returncode = process.wait()
except KeyboardInterrupt:
    process.terminate()
    process.wait(timeout=30)
    raise
if returncode:
    raise RuntimeError(f'Experiment stopped (exit {returncode}). See {log_path}; resume from the last saved update.')

Read the results: speedup above 1 is faster; memory and perplexity ratios below 1 are smaller/better.
`strict_quality_win` requires lower NLL than both original and adapted attention controls,
with the upper paired 95% interval below zero. `quality_within_margin` permits up to 0.02 nats
deterioration and must not be described as a quality win. Four-document smoke runs cannot
earn quality-win labels. The selected candidate is locked on validation before reading test results.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
report = json.loads((OUT/'report.json').read_text())
table = pd.DataFrame(report['rows'])
cols = ['candidate','context','test_nll','ppl_ratio','decode_speedup','prefill_speedup',
        'cache_ratio','peak_memory_ratio','strict_quality_win','meets_requested_target']
display(table.reindex(columns=cols))
print('Selected on validation:',report['selected'])
for context in sorted(table.context.unique()):
    part = table[table.context==context].set_index('candidate')
    fig,axes = plt.subplots(1,3,figsize=(16,4))
    for ax,metric,label in zip(axes,['ppl_ratio','decode_speedup','cache_ratio'],
                              ['Perplexity / original (lower better)','Decode speedup (higher better)','Cache / original (lower better)']):
        part[metric].plot.bar(ax=ax,color=['#64748b','#94a3b8','#2563eb','#0d9488','#a855f7'][:len(part)])
        ax.axhline(1,color='black',linestyle='--',linewidth=1)
        if metric=='decode_speedup':ax.axhline(2,color='#16a34a',linestyle=':',label='2× target')
        ax.set_title(label);ax.set_xlabel('');ax.tick_params(axis='x',rotation=30)
    fig.suptitle(f'Context {context} — complete model, batch one')
    fig.tight_layout()
    fig.savefig(OUT/f'comparison-T{context}.png',dpi=160,bbox_inches='tight')
    plt.show()
examples = json.loads((OUT/'generation_examples.json').read_text())
for example in examples:
    if example['candidate'] in ('original',report['selected']):
        print(chr(10),example['candidate'],'|',example['prompt'],chr(10),example['text'])

Optional: use the selected checkpoint for a custom prompt. It is an adapter plus the immutable
base-model revision recorded in its metadata, not a standalone Hugging Face model.

In [ ]:
RUN_CUSTOM_PROMPT = False
PROMPT = 'Explain recursion with a simple example.'
if RUN_CUSTOM_PROMPT:
    prompt_file = OUT/'custom_prompt.txt'
    prompt_file.write_text(PROMPT)
    # A separate process also ensures the GPU is released after generation.
    code_text = r"""
import json, sys, torch
from pathlib import Path
from types import SimpleNamespace
root, out = Path(sys.argv[1]), Path(sys.argv[2])
sys.path[:0] = [str(root),str(root/'src')]
from scripts.benchmark_nonlinear_readout import generate, dtype_for
from tinycenn_lm.nonlinear_readout import restore_student
from transformers import AutoModelForCausalLM, AutoTokenizer
report = json.loads((out/'report.json').read_text())
record = next(r for r in report['candidates'] if r['candidate']==report['selected'])
payload = torch.load(out/record['checkpoint'],map_location='cpu',weights_only=True)
metadata = payload['metadata']
family = json.loads((out/'manifest.json').read_text())['configuration']['family']
loader = AutoModelForCausalLM
if family=='qwen35':
    from transformers import Qwen3_5ForCausalLM
    loader=Qwen3_5ForCausalLM
tokenizer=AutoTokenizer.from_pretrained(metadata['base_model'],revision=metadata['model_revision'])
base=loader.from_pretrained(metadata['base_model'],revision=metadata['model_revision'],torch_dtype=dtype_for('cuda'),attn_implementation='sdpa')
model=restore_student(base,payload).cuda().eval()
del base
args=SimpleNamespace(family=family,device='cuda',generation_tokens=128)
print(generate(model,tokenizer,(out/'custom_prompt.txt').read_text(),args))
"""
    subprocess.run([sys.executable,'-c',code_text,str(REPO),str(OUT)],cwd=REPO,check=True)

In [ ]:
# Drive already contains the complete run. Download a ZIP only if wanted.
DOWNLOAD_ZIP = False
if DOWNLOAD_ZIP:
    archive = shutil.make_archive('/content/'+FAMILY+'-'+RUN_NAME,'zip',OUT)
    from google.colab import files
    files.download(archive)
print('Saved report:',OUT/'report.json')
print('To resume, keep RUN_NAME =',repr(RUN_NAME),'and set RESUME = True.')